# Lab 1：HP 產品規格查詢 Agent

## 學習目標
1. 寫一個**產品規格查詢工具**：輸入型號 → 回傳規格
2. 讓 **LLM 自己決定**要不要用工具（Tool Calling）
3. 測試兩種情境：查規格 vs 閒聊

## 流程
```
使用者問題 → LLM 判斷 → [需要規格] → 呼叫工具 → 回答
                      → [閒聊]    → 直接回答
```

In [ ]:
%%capture
!pip install langchain-openai langchain-core

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

os.environ["OPENAI_API_KEY"] = "your-api-key-here"

llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)
print("✓ 設定完成")

## Step 1：寫產品規格查詢工具

用 `@tool` 裝飾器把函數變成 LangChain Tool。  
LLM 會讀取 docstring 來了解這個工具的用途，所以描述要清楚。

In [ ]:
# HP 產品規格資料庫
HP_PRODUCTS = {
    "elitebook 855": {
        "名稱": "HP EliteBook 855 G10",
        "CPU": "AMD Ryzen 7 PRO 7840U",
        "RAM": "32GB DDR5",
        "儲存": "512GB PCIe NVMe SSD",
        "螢幕": "15.6 吋 FHD IPS",
        "重量": "1.74 kg",
    },
    "probook 450": {
        "名稱": "HP ProBook 450 G10",
        "CPU": "Intel Core i7-1355U",
        "RAM": "16GB DDR4",
        "儲存": "256GB PCIe NVMe SSD",
        "螢幕": "15.6 吋 FHD IPS",
        "重量": "1.79 kg",
    },
}

# =============================================
# TODO：完成這個工具函數（約 5 行）
#
# 提示：
#   1. 把 model_name 轉小寫，在 HP_PRODUCTS 裡查詢
#   2. 找到了 → 把規格 dict 格式化成字串回傳
#   3. 找不到 → 回傳「查無此型號」
# =============================================
@tool
def get_product_spec(model_name: str) -> str:
    """查詢 HP 筆電規格。輸入型號（如 EliteBook 855），回傳詳細規格。"""
    pass  # 請實作這裡


# 快速測試工具本身
print(get_product_spec.invoke({"model_name": "EliteBook 855"}))

## Step 2：讓 LLM 自己決定要不要用工具

`bind_tools()` 把工具清單告訴 LLM。  
LLM 收到問題後，**自己判斷**：
- 需要查規格 → 回傳 `tool_calls`，我們執行工具再把結果餵回去
- 閒聊問題 → 直接回傳文字答案

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# 把工具綁定給 LLM
llm_with_tools = llm.bind_tools([get_product_spec])

def run_agent(question: str) -> str:
    """簡單 Agent：LLM 決定要不要查工具，最多執行一次工具。"""
    messages = [HumanMessage(content=question)]

    # 第一輪：LLM 判斷
    response = llm_with_tools.invoke(messages)

    # 沒有工具呼叫 → 直接回答
    if not response.tool_calls:
        return response.content

    # 有工具呼叫 → 執行工具，把結果餵回 LLM
    messages.append(response)
    for tc in response.tool_calls:
        result = get_product_spec.invoke(tc["args"])
        messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

    final = llm_with_tools.invoke(messages)
    return final.content

print("✓ Agent 準備好了")

## Step 3：測試

In [ ]:
# 測試 1：應該走工具
print("=" * 50)
print("問：EliteBook 855 有多少 RAM？")
print("=" * 50)
print(run_agent("EliteBook 855 有多少 RAM？"))

print()

# 測試 2：應該直接回答
print("=" * 50)
print("問：你好")
print("=" * 50)
print(run_agent("你好"))